# Indic IT Incident Triage with Sarvam-105B

Turn Hindi, Tamil and English IT-support messages into structured JSON.

**Pipeline:** incident text → Sarvam Chat Completions (`response_format=json_schema`) → Pydantic validation → JSON file. This is a demonstration, not autonomous remediation.

In [ ]:
%pip install -r requirements.txt

## Configure a private API key

Copy `.env.example` to `.env`, fill in `SARVAM_API_KEY`, and keep `.env` out of version control. See [Sarvam authentication](https://docs.sarvam.ai/api-reference-docs/authentication).

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

from incident_triage import classify_incident

load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError("Set SARVAM_API_KEY in your environment or .env file.")

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Classify multilingual synthetic incidents

The model returns a constrained JSON shape. The Pydantic model adds stricter application checks. Sample inputs contain no personal information.

In [ ]:
sample_reports = [
    "कार्यालय का वाई-फ़ाई दो घंटे से बंद है। सबका काम रुका हुआ है।",
    "எனது மடிக்கணினியில் மென்பொருள் திறக்கவில்லை.",
    "I cannot sign in to my work account after resetting the password.",
]

results = [classify_incident(client, report).model_dump() for report in sample_reports]
for result in results:
    print(json.dumps(result, ensure_ascii=False, indent=2))

## Save and inspect the structured output

This writes only the synthetic demonstration results. Review `urgency` and missing details before acting on real-world data.

In [ ]:
output_file = OUTPUT_DIR / "example_triage.json"
output_file.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(results)} synthetic records to {output_file}")